# Homework 6 Coding Part 2 - DQN + Double DQN in JAX
In this assignment, we will solve a miniature version of the classical atari game [Asterix](https://ale.farama.org/environments/asterix/).

We will be using [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) to just-in-time (JIT) compile the entire training loop.

One major advantage of JAX is that the compiler supports the TPU backend. Due to this, it is strongly encouraged to use Colab with a v5e1-TPU instance.

Many of the provided repos for the course project are written in JAX, so it would be a good idea to understand the training code even though it is not necessary to complete the assignment.

### Formulation

- *State* $s$:
    Each game in the MinAtar suite consists of n channels of 10x10 grids. For asterix, there are four channels

|Num|Observation
|:-:|:-:
|0|Player
|1|Enemy
|2|Trail
|3|Gold

- *Action $a$*:
    There are five possible actions as follows:

|Num|Action|
|:-:|:-:|
|0|Noop|
|1|Left|
|2|Right|
|3|Up|
|4|Down|


- *Reward $r(s,a)$*:  Reward is either 1 or 0 for every step taken, 1 being when the agent picks up treasure.
    

- *Episode Termination*: Termination occurs if the player makes contact with an enemy.

- *Objective*: Well-tuned PPO achieves an average award of ~50. For this assignment, you should achieve > 30


In [ ]:
!pip -q install -U gymnax distrax optax flax flashbax chex

In [ ]:
import gymnax
from gymnax.wrappers.purerl import FlattenObservationWrapper, LogWrapper

basic_env, env_params = gymnax.make("Asterix-MinAtar")
env = FlattenObservationWrapper(basic_env)
env = LogWrapper(env)

In [ ]:
from abc import abstractmethod
import chex

import flax
import flax.linen as nn
import flax.core
from flax.training.train_state import TrainState

import flashbax as fbx
import jax
import jax.numpy as jnp
import optax



class QNetwork(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, x: jnp.ndarray):
        x = nn.Dense(120)(x)
        x = nn.relu(x)
        x = nn.Dense(84)(x)
        x = nn.relu(x)
        x = nn.Dense(self.action_dim)(x)
        return x


@chex.dataclass(frozen=True)
class TimeStep:
    obs: chex.Array
    action: chex.Array
    reward: chex.Array
    done: chex.Array


class CustomTrainState(TrainState):
    target_network_params: flax.core.FrozenDict
    timesteps: int
    n_updates: int


In [ ]:
@chex.dataclass
class DQNBase:
    num_envs: int = 32
    buffer_size: int = 10_000
    buffer_batch_size: int = 32
    total_timesteps: float = 1e7
    epsilon_start: float = 1.0
    epsilon_finish: float = 0.05
    epsilon_anneal_time: float = 25e4
    target_update_interval: int = 500
    lr: float = 2.5e-4
    learning_starts: int = 10_000
    training_interval: int = 10
    gamma: float = 0.99
    tau: float = 1.0
    seed: int = 0

    def linear_schedule(self, count):
        frac = 1.0 - (count / self.num_updates)
        return self.lr * frac

    def epsilon_at_t(self, t):
        eps = ((self.epsilon_finish - self.epsilon_start) / self.epsilon_anneal_time) * t + self.epsilon_start
        return jnp.clip(eps, self.epsilon_finish)

    def select_action(self, rng, q_vals, t):
        rng_a, rng_e = jax.random.split(rng, 2)
        eps = self.epsilon_at_t(t)
        greedy_actions = jnp.argmax(q_vals, axis=-1)
        random_actions = jax.random.randint(
            rng_a, shape=greedy_actions.shape, minval=0, maxval=q_vals.shape[-1]
        )
        do_random = jax.random.uniform(rng_e, greedy_actions.shape) < eps
        return jnp.where(do_random, random_actions, greedy_actions)

    def init_train_state(self, rng):
        action_dim = env.action_space(env_params).n
        network = QNetwork(action_dim=action_dim)

        rng, init_rng = jax.random.split(rng)
        init_x = jnp.zeros(env.observation_space(env_params).shape)
        params = network.init(init_rng, init_x)

        tx = optax.adam(learning_rate=self.lr)

        train_state = CustomTrainState.create(
            apply_fn=network.apply,
            params=params,
            target_network_params=jax.tree_util.tree_map(lambda x: jnp.array(x, copy=True), params),
            tx=tx,
            timesteps=0,
            n_updates=0,
        )
        return network, train_state


    def build_buffer(self):
        buffer = fbx.make_flat_buffer(
            max_length=self.buffer_size,
            min_length=self.buffer_batch_size,
            sample_batch_size=self.buffer_batch_size,
            add_sequences=False,
            add_batch_size=self.num_envs,
        )

        rng0 = jax.random.PRNGKey(0)
        dummy_action = basic_env.action_space().sample(rng0)
        _, dummy_state = env.reset(rng0, env_params)
        dummy_obs, _, dummy_reward, dummy_done, _ = env.step(rng0, dummy_state, dummy_action, env_params)
        dummy_ts = TimeStep(obs=dummy_obs, action=dummy_action, reward=dummy_reward, done=dummy_done)
        buffer_state = buffer.init(dummy_ts)
        return buffer, buffer_state


    @abstractmethod
    def compute_target(self, network, train_state, batch):
        pass


    def learn_step(self, network, train_state, buffer, buffer_state, rng):
        batch = buffer.sample(buffer_state, rng).experience
        target = self.compute_target(network, train_state, batch)

        def loss_fn(params):
            q = network.apply(params, batch.first.obs)  # (B, A)
            q_chosen = jnp.take_along_axis(
                q, batch.first.action[:, None], axis=-1
            ).squeeze(-1)  # (B,)
            return jnp.mean((q_chosen - target) ** 2)

        loss, grads = jax.value_and_grad(loss_fn)(train_state.params)
        train_state = train_state.apply_gradients(grads=grads)
        train_state = train_state.replace(n_updates=train_state.n_updates + 1)
        return train_state, loss


    def maybe_update_target(self, train_state):
        new_target = jax.lax.cond(
            train_state.timesteps % self.target_update_interval == 0,
            lambda train_state: optax.incremental_update(
                train_state.params, train_state.target_network_params, self.tau
            ),
            lambda train_state: train_state.target_network_params,
            train_state
        )
        return train_state.replace(target_network_params=new_target)


    def train(self, rng):
        num_updates = int(self.total_timesteps // self.num_envs)

        # Vectorized reset/step (these will be traced inside the jitted scan)
        def vmap_reset(rng):
            return jax.vmap(env.reset, in_axes=(0, None))(
                jax.random.split(rng, self.num_envs), env_params
            )

        def vmap_step(rng, env_state, action):
            return jax.vmap(env.step, in_axes=(0, 0, 0, None))(
                jax.random.split(rng, self.num_envs), env_state, action, env_params
            )

        # Init env
        rng, r0 = jax.random.split(rng)
        obs0, env_state0 = vmap_reset(r0)

        # Buffer + network/state
        buffer, buffer_state0 = self.build_buffer()
        network, train_state0 = self.init_train_state(rng)

        # ---- Make an update after a batched env step ----
        def one_step(carry, _):
            rng, obs, env_state, train_state, buffer_state, ema_carry = carry

            rng, r_a, r_s, r_l = jax.random.split(rng, 4)

            # Policy action
            q_vals = train_state.apply_fn(train_state.params, obs)  # (N, A)
            action = self.select_action(r_a, q_vals, train_state.timesteps)

            # Env step (batched)
            next_obs, env_state, reward, done, info = vmap_step(r_s, env_state, action)

            # Add to buffer
            ts = TimeStep(obs=obs, action=action, reward=reward, done=done)
            buffer_state = buffer.add(buffer_state, ts)

            # Increment timesteps
            train_state = train_state.replace(
                timesteps=train_state.timesteps + jnp.int32(self.num_envs)
            )

            # Warm-up before learning
            do_learn = (train_state.timesteps >= jnp.int32(self.learning_starts)) & (
                (train_state.timesteps % jnp.int32(self.training_interval)) == 0
            )

            def learn_branch(ts_in):
                ts_out, loss = self.learn_step(network, ts_in, buffer, buffer_state, r_l)
                return ts_out, loss

            def no_learn_branch(ts_in):
                return ts_in, jnp.array(0.0, dtype=jnp.float32)

            train_state, loss = jax.lax.cond(do_learn, learn_branch, no_learn_branch, train_state)

            # Target update
            train_state = self.maybe_update_target(train_state)

            ep_ret = jnp.mean(info["returned_episode_returns"]).astype(jnp.float32)
            ema_ret = 0.99 * ema_carry + 0.01 * ep_ret

            metrics_t = (
                train_state.timesteps.astype(jnp.int32),
                train_state.n_updates.astype(jnp.int32),
                loss.astype(jnp.float32),
                ep_ret,
                ema_ret
            )

            new_carry = (rng, next_obs, env_state, train_state, buffer_state, ema_ret)
            return new_carry, metrics_t

        # Perform num_updates number of updates
        def update(rng, obs, env_state, train_state, buffer_state):
            carry0 = (rng, obs, env_state, train_state, buffer_state, 0.0)
            carryN, metrics = jax.lax.scan(one_step, carry0, xs=None, length=num_updates)
            return carryN, metrics

        (rngN, obsN, env_stateN, train_stateN, buffer_stateN, final_ema), metrics = update(
            rng, obs0, env_state0, train_state0, buffer_state0
        )

        timesteps, updates, losses, returns, ema_returns = metrics
        metrics_dict = {
            "timesteps": timesteps,
            "updates": updates,
            "loss": losses,
            "returns": returns,
            "ema_returns": ema_returns
        }
        return {"train_state": train_stateN, "buffer_state": buffer_stateN, "metrics": metrics_dict}

# DQN
Look at the following implementation of DQN to see how to access the necessary Q values / observations from the batch generated by the replay buffer.

In [ ]:
def dqn_target_from_q(
    q_next_target: jnp.ndarray,   # (B, A)
    reward: jnp.ndarray,          # (B,)
    done: jnp.ndarray,            # (B,)
    gamma: float,
) -> jnp.ndarray:
    ### BEGIN SOLUTION
    # YOUR CODE HERE
    q_next_max = jnp.max(q_next_target, axis=-1)
    target = reward + gamma * (1.0 - done) * q_next_max
    #raise NotImplementedError()
    ### END SOLUTION
    return jax.lax.stop_gradient(target)


class DQN(DQNBase):
    def compute_target(self, network, train_state, batch):
        '''
        args:
          network: module that contains forward pass via network.apply(params, obs)
          train_state: contains parameters for online (train_state.params) and target (train_state.target)
          batch: contains TimeStep batch with s as batch.first.obs and s' as batch.second.obs
        '''
        ### BEGIN SOLUTION
        # YOUR CODE HERE
        q_next_target = network.apply(train_state.target_network_params, batch.second.obs)

        #raise NotImplementedError()
        ### END SOLUTION
        return dqn_target_from_q(
            q_next_target,
            batch.first.reward, batch.first.done,
            self.gamma,
        )

In [ ]:
# For debugging, comment out before submitting.

import time

rng = jax.random.PRNGKey(42)
dqn_agent = DQN()

train = jax.jit(dqn_agent.train)

# ---- compile time only ----
t0 = time.time()
compiled = train.lower(rng).compile()
t1 = time.time()
print("Compile time:", t1 - t0)

# ---- execution time only ----
t2 = time.time()
dqn_out = train(rng)
jax.tree_util.tree_map(lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x, dqn_out)
t3 = time.time()
print("Train time:", t3 - t2)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(dqn_out["metrics"]["updates"], dqn_out["metrics"]["ema_returns"])
plt.xlabel("Updates")
plt.ylabel("Return")
plt.show()

# Double DQN
Now to avoid the overestimation of the traditional DQN update. You must recompute the target following the update in the [Double DQN Paper](https://arxiv.org/pdf/1509.06461).

In [ ]:
def double_dqn_target_from_q(
    q_next_online: jnp.ndarray,   # (B, A)
    q_next_target: jnp.ndarray,   # (B, A)
    reward: jnp.ndarray,          # (B,)
    done: jnp.ndarray,            # (B,)
    gamma: float,
) -> jnp.ndarray:
    ### BEGIN SOLUTION
    # YOUR CODE HERE
    q_next_max = jnp.take_along_axis(q_next_target, 
                                     jnp.argmax(q_next_online, axis=-1)[:, None], 
                                     axis=-1).squeeze(-1)
    target = reward + gamma * (1.0 - done) * q_next_max
    #raise NotImplementedError()
    ### END SOLUTION
    return jax.lax.stop_gradient(target)


class DoubleDQN(DQNBase):
    def compute_target(self, network, train_state, batch):
        '''
        args:
          network: module that contains forward pass via network.apply(params, obs)
          train_state: contains parameters for online (train_state.params) and target (train_state.target)
          batch: contains TimeStep batch with s as batch.first.obs and s' as batch.second.obs
        '''
        ### BEGIN SOLUTION
        # YOUR CODE HERE
        q_next_online = network.apply(train_state.params, batch.second.obs)
        q_next_target = network.apply(train_state.target_network_params, batch.second.obs)
        #raise NotImplementedError()
        ### END SOLUTION
        return double_dqn_target_from_q(
            q_next_online, q_next_target,
            batch.first.reward, batch.first.done,
            self.gamma,
        )

In [ ]:
# For debugging, comment out before submitting.

rng = jax.random.PRNGKey(42)
ddqn_agent = DoubleDQN()

train = jax.jit(ddqn_agent.train)

# ---- compile time only ----
t0 = time.time()
compiled = train.lower(rng).compile()
t1 = time.time()
print("Compile time:", t1 - t0)

# ---- execution time only ----
t2 = time.time()
ddqn_out = train(rng)
jax.tree_util.tree_map(lambda x: x.block_until_ready() if hasattr(x, "block_until_ready") else x, ddqn_out)
t3 = time.time()
print("Train time:", t3 - t2)

In [ ]:
plt.plot(ddqn_out["metrics"]["updates"], ddqn_out["metrics"]["ema_returns"])
plt.xlabel("Updates")
plt.ylabel("Return")
plt.show()

In [ ]:
from pathlib import Path
import flax

def save_params(params, path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(flax.serialization.to_bytes(params))

print(ddqn_out['train_state'].params)
save_params(ddqn_out['train_state'].params['params'], "ddqn.msgpack")